# Points vs budget: finding the ideal exchange rate

The optimiser currently maximises expected fantasy points only. Because budget carries
forward as `cash + team value`, an asset that appreciates increases future buying power
and lets us afford stronger drivers sooner. This notebook searches for the ideal
trade-off between **points now** and **future buying power**.

We add a term to the ILP objective:

> maximise  expected_points  +  **lambda** x expected_next_round_price_gain

`lambda` is an exchange rate: fantasy points valued per GBP1M of future buying power.
We replay the full 2026 backtest at each `lambda` and read off the **cumulative season
points** it produces, so the winner is judged on the metric that actually matters.

Two experiments:
1. **Continuous lambda sweep** - one constant lambda all season.
2. **Two-phase farm-then-spend** - farm price for the first N rounds, then go points-greedy.

Note: price deltas here are the *realised* next-round changes (perfect foresight of price
moves). Since prices are deterministic from points, this is close to attainable live, but
treat the resulting lambda as an upper-bound-flavoured estimate - real prediction error on
points would erode some of the edge.

## Setup
Load models and predictors once (the slow part).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, pulp, fastf1
import matplotlib.pyplot as plt

from app.config import (PROCESSED_PRICES_DIR, PROCESSED_HISTORIC_FEATURES_DIR, PROCESSED_TARGETS_DIR,
                        INTERIM_EVENTS_DIR, BUDGET_CAP, DRIVER_ROSTER_SIZE, CONSTRUCTOR_ROSTER_SIZE)
from app.models.configs import FINISH_POSITION_MODEL, QUALI_POSITION_MODEL
from app.models.compose import compose_drivers, compose_constructor, expected_pitstop_points
from app.interface.cli import run_predict, load_season_model, _build_state
from app.data.overtakes import build_overtake_predictor
from app.data.dotd import build_dotd_predictor
from app.backtest import get_actual_team_points

SEASON = 2026
BUDGET = BUDGET_CAP  # season-start budget cap

predict_overtakes = build_overtake_predictor()
predict_dotd = build_dotd_predictor()
quali_model = load_season_model(QUALI_POSITION_MODEL, SEASON)
finish_model = load_season_model(FINISH_POSITION_MODEL, SEASON)
print("models + predictors loaded")

## Precompute per-round data
Run predictions once per round and cache points, prices and the realised next-round
price delta. Every lambda then reuses this cache, so only the fast ILP re-solves.

In [ ]:
schedule = fastf1.get_event_schedule(SEASON)
schedule = schedule[schedule["RoundNumber"] > 0]

valid_rounds = []
for _, e in schedule.iterrows():
    rn = int(e["RoundNumber"])
    if all(p.exists() for p in [PROCESSED_PRICES_DIR/f"{SEASON}_{rn:02d}.parquet",
                                PROCESSED_HISTORIC_FEATURES_DIR/f"{SEASON}_{rn:02d}.parquet",
                                PROCESSED_TARGETS_DIR/f"{SEASON}_{rn:02d}.parquet"]):
        valid_rounds.append((rn, e.get("Location", str(rn))))
print("valid rounds:", [r for r, _ in valid_rounds])

def load_price(rn):
    p = PROCESSED_PRICES_DIR/f"{SEASON}_{rn:02d}.parquet"
    return pd.read_parquet(p).set_index("asset_id")["price"] if p.exists() else None

cache = []
for rn, loc in valid_rounds:
    prices = pd.read_parquet(PROCESSED_PRICES_DIR/f"{SEASON}_{rn:02d}.parquet")
    ap = prices.set_index("asset_id")["price"]
    ep = INTERIM_EVENTS_DIR/f"{SEASON}_{rn:02d}.parquet"
    btloc = pd.read_parquet(ep)["location"].iloc[0] if ep.exists() else None
    preds = run_predict(quali_model, QUALI_POSITION_MODEL, finish_model, FINISH_POSITION_MODEL, SEASON, rn)
    dp = compose_drivers(preds, location=btloc, season=SEASON, predict_overtakes=predict_overtakes, predict_dotd=predict_dotd)
    cp = compose_constructor(dp, pitstop_pts=expected_pitstop_points(SEASON, rn))
    # realised next-round price delta = value gained by holding an asset from this round into the next
    nxt = load_price(rn + 1)
    delta = {a: (float(nxt.get(a, ap[a])) - float(ap[a])) if nxt is not None else 0.0 for a in ap.index}
    cache.append(dict(round=rn, location=loc, prices=prices, asset_prices=ap,
                      driver_points=dp, constructor_points=cp, price_delta=delta))
print(f"cached {len(cache)} rounds")

## Price-aware optimiser
The stock optimiser plus one term: `lambda * expected next-round price gain` for each
selected asset. Everything else (roster sizes, doubled driver, transfer penalty, budget
constraint) is unchanged.

In [ ]:
def price_aware_optimiser(driver_points, constructor_points, prices, price_delta, lam=0.0, budget=BUDGET, state=None):
    prob = pulp.LpProblem("f1_fantasy_price_aware", pulp.LpMaximize)
    prices_index = prices.set_index("asset_id")["price"]
    priced = set(prices_index.index)

    drivers = [d for d in driver_points["driver_id"] if d in priced]
    constructors = [c for c in constructor_points["constructor_id"] if c in priced]
    dpts = driver_points.set_index("driver_id")["expected_fantasy_points"]
    cpts = constructor_points.set_index("constructor_id")["expected_fantasy_points"]

    if state is not None:
        prev_team = set(state["drivers"] + state["constructors"])
        free_transfers = 2 + state["free_transfers_carried"]
        available = state["budget_remaining"]
        for i in prev_team:
            available += prices_index[i] if i in prices_index else state["prices"][i]
    else:
        prev_team, free_transfers, available = set(), 0, budget

    selected = pulp.LpVariable.dicts("selected", drivers + constructors, cat="Binary")
    doubled = pulp.LpVariable.dicts("doubled", drivers, cat="Binary")
    penalty = pulp.LpVariable("penalty", lowBound=0)

    prob += (
        pulp.lpSum(dpts[d]*selected[d] for d in drivers)
        + pulp.lpSum(cpts[c]*selected[c] for c in constructors)
        + pulp.lpSum(dpts[d]*doubled[d] for d in drivers)                       # doubled driver scores twice
        + lam * pulp.lpSum(price_delta.get(i, 0.0)*selected[i] for i in drivers + constructors)
        - 10*penalty
    )

    prob += pulp.lpSum(selected[d] for d in drivers) == DRIVER_ROSTER_SIZE
    prob += pulp.lpSum(selected[c] for c in constructors) == CONSTRUCTOR_ROSTER_SIZE
    prob += pulp.lpSum(doubled[d] for d in drivers) == 1
    for d in drivers:
        prob += doubled[d] <= selected[d]
    prob += (pulp.lpSum(prices_index[d]*selected[d] for d in drivers)
             + pulp.lpSum(prices_index[c]*selected[c] for c in constructors)) <= available
    if state is not None:
        transfers_in = pulp.lpSum(selected[i] for i in drivers + constructors if i not in prev_team)
        prob += penalty >= transfers_in - free_transfers

    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    sd = [d for d in drivers if pulp.value(selected[d]) == 1]
    sc = [c for c in constructors if pulp.value(selected[c]) == 1]
    dd = [d for d in drivers if pulp.value(doubled[d]) == 1][0]
    return {"drivers": sd, "constructors": sc, "doubled_driver": dd,
            "transfers_made": sum(1 for i in sd+sc if i not in prev_team),
            "transfer_penalty": round(pulp.value(penalty)), "dropped": []}

Season replay: apply a lambda schedule, carry budget forward, score on **actual**
points. `farm_rounds > 0` runs the two-phase split - the first `farm_rounds` use
`farm_lam`, the rest use `spend_lam`.

In [ ]:
def replay(spend_lam=0.0, farm_rounds=0, farm_lam=0.0):
    state, cum, rows = None, 0.0, []
    for idx, c in enumerate(cache):
        lam = farm_lam if idx < farm_rounds else spend_lam
        team = price_aware_optimiser(c["driver_points"], c["constructor_points"], c["prices"],
                                     c["price_delta"], lam=lam, budget=BUDGET, state=state)
        pts = get_actual_team_points(team, SEASON, c["round"], team["transfer_penalty"])
        cum += pts
        state = _build_state(team, state, c["asset_prices"], BUDGET)
        spend_power = state["budget_remaining"] + sum(c["asset_prices"][i] for i in team["drivers"]+team["constructors"])
        rows.append(dict(round=c["round"], lam=lam, points=pts, cum_points=round(cum,1),
                         team_cost=round(sum(c["asset_prices"][i] for i in team["drivers"]+team["constructors"]),1),
                         spend_power=round(spend_power,1), transfers=team["transfers_made"]))
    return pd.DataFrame(rows)

# sanity check: lambda=0 should reproduce the stock points-greedy backtest
_base = replay(spend_lam=0.0)
print("baseline cumulative points (lambda=0):", _base["cum_points"].iloc[-1])
print("baseline final buying power:", _base["spend_power"].iloc[-1])
_base

## Section 1: continuous lambda sweep
One constant lambda for the whole season. We want the lambda that maximises cumulative
season points, not the one that maximises budget.

In [ ]:
lambdas = [0, 0.5, 1, 2, 3, 5, 8, 12, 20, 40, 70, 120, 250, 500]
sweep = pd.DataFrame([
    dict(lam=lam, **{"cum_points": (df:=replay(spend_lam=lam))["cum_points"].iloc[-1],
                     "final_buying_power": df["spend_power"].iloc[-1]})
    for lam in lambdas
])
sweep

In [ ]:
fig, ax1 = plt.subplots(figsize=(8,5))
ax1.plot(sweep["lam"], sweep["cum_points"], "o-", color="tab:blue", label="cumulative points")
ax1.set_xlabel("lambda  (points valued per GBP1M future buying power)")
ax1.set_ylabel("cumulative season points", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(sweep["lam"], sweep["final_buying_power"], "s--", color="tab:green")
ax2.set_ylabel("final buying power (GBP M)", color="tab:green")
best = sweep.loc[sweep["cum_points"].idxmax()]
ax1.axvline(best["lam"], color="grey", ls=":", alpha=0.8)
plt.title(f"Points vs budget - best lambda = {best['lam']}  ({best['cum_points']:.0f} pts)")
fig.tight_layout(); plt.show()

baseline = sweep.loc[sweep["lam"]==0].iloc[0]
print(f"baseline (lambda=0):   {baseline['cum_points']:.0f} pts, buying power GBP{baseline['final_buying_power']:.1f}M")
print(f"best     (lambda={best['lam']}): {best['cum_points']:.0f} pts, buying power GBP{best['final_buying_power']:.1f}M")
print(f"delta vs baseline: {best['cum_points']-baseline['cum_points']:+.0f} pts")

## Section 2: two-phase farm-then-spend
Farm price aggressively for the first N rounds (`farm_lam`), then switch to pure
points (`spend_lam = 0`). Sweep over both to see if front-loading budget beats a
constant lambda. Column 0 (zero farm rounds) is the points-greedy baseline.

In [ ]:
farm_rounds_grid = list(range(0, len(cache)))     # early rounds spent farming
farm_lams = [5, 20, 40, 70, 120]                  # farm intensity

grid = np.zeros((len(farm_lams), len(farm_rounds_grid)))
for i, fl in enumerate(farm_lams):
    for j, n in enumerate(farm_rounds_grid):
        grid[i, j] = replay(spend_lam=0.0, farm_rounds=n, farm_lam=fl)["cum_points"].iloc[-1]
print("done")

In [ ]:
fig, ax = plt.subplots(figsize=(11,4))
im = ax.imshow(grid, aspect="auto", cmap="viridis", origin="lower")
ax.set_xticks(range(len(farm_rounds_grid))); ax.set_xticklabels(farm_rounds_grid)
ax.set_yticks(range(len(farm_lams))); ax.set_yticklabels(farm_lams)
ax.set_xlabel("farm rounds (early rounds spent farming price)")
ax.set_ylabel("farm lambda")
fig.colorbar(im, label="cumulative season points")
bi = np.unravel_index(np.argmax(grid), grid.shape)
ax.plot(bi[1], bi[0], "r*", markersize=20)
best_pts = grid[bi]; base_pts = grid[:,0].max()
plt.title(f"Farm-then-spend - best: {farm_rounds_grid[bi[1]]} farm rounds @ lambda {farm_lams[bi[0]]} = {best_pts:.0f} pts")
plt.tight_layout(); plt.show()

print(f"points-greedy baseline (0 farm rounds): {base_pts:.0f} pts")
print(f"best farm-then-spend:                   {best_pts:.0f} pts  "
      f"({farm_rounds_grid[bi[1]]} rounds @ lambda {farm_lams[bi[0]]})")
print(f"delta vs baseline: {best_pts-base_pts:+.0f} pts")

## Findings (2026 backtest)
**Price-awareness is a large, real effect - worth wiring in.**

**Section 1 - constant lambda sweep.** There is a clear interior optimum around
**lambda ~= 40**:

| strategy | cumulative points | final buying power |
|---|---|---|
| points-greedy (lambda=0) | 2487 | GBP105.7M |
| best (lambda~=40) | **2868 (+381, +15%)** | GBP130.3M |

Points climb with lambda up to ~40, then fall away; by lambda>=250 the objective ignores
points entirely and cumulative score drops *below* the baseline (~1820). So farming has a
sweet spot - too much and you sacrifice real scoring for paper value. At the optimum,
buying power grows to ~GBP130M, matching the intuition that deliberate farming reaches
>GBP120M by round 11.

**Section 2 - farm-then-spend.** Best split was 10 farm rounds @ lambda 40 = 2844 pts,
which does **not** beat the constant lambda=40 (2868). A single blended objective all
season is at least as good as an explicit farm-then-spend phase here, and simpler.

**Big caveat - this is an upper bound.** The price deltas above are the *realised*
next-round moves (perfect foresight). Live, the delta must be estimated from *predicted*
points via the rolling-PPM rule in `app/data/prices.py`, so the real edge - and the
actionable lambda - will be smaller. The +381 pts is the ceiling, not the expectation.

**Recommended next step.** Add a `price_delta_predicted` variant (rolling PPM on predicted
points), re-run this sweep, and compare its optimum to the realised-delta ceiling. If the
predicted version still clears the baseline by a worthwhile margin, wire the lambda term
into `app/optimiser/optimiser.py` behind a tunable weight. If it collapses to ~0, record
it as 'explored, deferred' in V3 - same as the DNF model.